In [ ]:
# notebooks/exploration.ipynb
import os
import sys
import yaml
import numpy as np
from nuscenes.nuscenes import NuScenes
import k3d # For K3D specific visualizations

# Add project root to sys.path
notebook_path = os.path.abspath(os.getcwd())
project_root = os.path.dirname(notebook_path)
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

%load_ext autoreload
%autoreload 2

from src.core import MDetector, OcclusionResult
from src.data_utils.nuscenes_helper import get_lidar_sweep_data
# from src.utils.visualization import plot_occlusion_test_results_k3d # If you refactor K3D plot
# from scripts.generate_mdetector_bev_video import generate_video # If you want to run video from notebook

# --- Load Config and Initialize NuScenes ---
config_file_path = os.path.join(project_root, 'config/m_detector_config.yaml')
with open(config_file_path, 'r') as f:
    config = yaml.safe_load(f)

nusc = NuScenes(version=config['nuscenes']['version'],
                dataroot=config['nuscenes']['dataroot'],
                verbose=False)

m_detector = MDetector(config=config)

scene_token = nusc.scene[1]['token'] # Example: first scene
current_sample_token = nusc.get('scene', scene_token)['first_sample_token']
lidar_sensor_name = config['nuscenes']['lidar_sensor_name']

# Determine how many sweeps to process for populating and labeling
num_sweeps_to_populate = config['initialization']['num_initial_sweeps_for_map'] + 10 # e.g., 5 for init + 10 more


In [ ]:

print(f"Populating M-Detector library with {num_sweeps_to_populate} sweeps...")
processed_sweeps_count = 0
temp_lidar_token = current_sample_token # Use a temp token for this loop
while temp_lidar_token and processed_sweeps_count < num_sweeps_to_populate:
    sample = nusc.get('sample', temp_lidar_token)
    lidar_token = sample['data'][lidar_sensor_name]
    points_lidar_frame, T_global_lidar, _ = get_lidar_sweep_data(nusc, lidar_token)
    lidar_sample_data_for_ts = nusc.get('sample_data', lidar_token)
    m_detector.add_sweep_and_create_depth_image(
        points_lidar_frame, T_global_lidar, lidar_sample_data_for_ts['timestamp']
    )
    processed_sweeps_count += 1
    if not sample['next']: break
    temp_lidar_token = sample['next']
print(f"M-Detector library populated with {len(m_detector.depth_image_library)} DIs.")



In [ ]:

# --- 2. Process and Assign Labels to Points in DepthImages ---
print("\n--- Processing and Assigning Labels to Points in DIs ---")
if m_detector.is_ready_for_processing():
    library = m_detector.depth_image_library
    # Determine historical lag (e.g., from config or fixed)
    historical_lag_indices = config.get('m_detector_processing', {}).get('historical_lag_indices', 1)
    
    # Iterate through DIs that can be processed (i.e., have a historical DI available)
    # The first DI that can be meaningfully processed is at index `historical_lag_indices`.
    num_dis_processed_for_labels = 0
    for i in range(historical_lag_indices, len(library)):
        current_di_to_label = library.get_image_by_index(i)
        historical_ref_di = library.get_image_by_index(i - historical_lag_indices)
        
        if current_di_to_label and historical_ref_di:
            print(f"Labeling DI {i} (TS: {current_di_to_label.timestamp/1e6:.2f}s) "
                  f"vs Hist. DI {i - historical_lag_indices} (TS: {historical_ref_di.timestamp/1e6:.2f}s)")
            labeled_count = m_detector.process_and_label_di(current_di_to_label, historical_ref_di)
            print(f"  Labeled {labeled_count} points in DI {i}.")
            num_dis_processed_for_labels +=1
        elif current_di_to_label: # No valid historical_ref_di
            print(f"Labeling DI {i} (TS: {current_di_to_label.timestamp/1e6:.2f}s) without historical reference (UNDETERMINED).")
            labeled_count = m_detector.process_and_label_di(current_di_to_label, None)
            print(f"  Labeled {labeled_count} points in DI {i} as UNDETERMINED.")
            num_dis_processed_for_labels +=1

    print(f"Label processing complete for {num_dis_processed_for_labels} DIs.")
else:
    print("M-Detector not ready for processing labels (library too small or initialization not met).")



In [ ]:

# --- 3. Visualize a Specific DepthImage with its Processed Labels ---
from src.utils.visualization import plot_predictions_k3d # Ensure this import is present

# Choose which DI to plot (e.g., the latest one for which labels were computed)
# Ensure the chosen index has actually been processed in step 2.
if len(m_detector.depth_image_library) > historical_lag_indices:
    # Example: Plot the latest DI that had its labels processed.
    # If historical_lag_indices = 1, the DIs from index 1 onwards were processed.
    # The last processed DI is at index: len(library) - 1
    index_to_plot_labels = len(m_detector.depth_image_library) - 1
    # Or choose another index that you know was processed, e.g., `historical_lag_indices` itself
    # index_to_plot_labels = historical_lag_indices 

    depth_image_for_viz = m_detector.depth_image_library.get_image_by_index(index_to_plot_labels)

    if depth_image_for_viz:
        print(f"\n--- Generating K3D plot for Labeled DI at library index {index_to_plot_labels} ---")
        
        plot = plot_predictions_k3d(
            depth_image_with_labels=depth_image_for_viz,
            point_size=0.08,
            plot_title=f"Labeled Points for DI {index_to_plot_labels} (TS: {depth_image_for_viz.timestamp/1e6:.2f}s)"
        )

        if plot:
            plot.display()
        else:
            print(f"Failed to generate K3D plot for DI at index {index_to_plot_labels}.")
    else:
        print(f"Could not retrieve DI at index {index_to_plot_labels} for plotting.")
else:
    print("Not enough DIs in the library (or processed) to select one for plotting with labels.")

